In [ ]:
# compare annual_inc across verification_status
import pandas as pd, numpy as np, json
df = pd.read_parquet("../data/clean_v2.parquet")
CUT = pd.Timestamp("2014-12-31"); fit = df[df.issue_d <= CUT].copy()
fit["round_inc"] = fit.annual_inc % 5000 == 0
g = fit.groupby("verification_status")
pd.DataFrame({"loans": g.size(), "share": g.size() / len(fit), "median_inc": g.annual_inc.median(),
              "p95_inc": g.annual_inc.quantile(.95), "round_5k_share": g.round_inc.mean(), "default_rate": g.defaulted.mean()}).round(3)

In [ ]:
# Assess borrower repayment burden and financial risk relationships across income-verification categories.
fit["pti"] = fit.installment * 12 / fit.annual_inc
fit["imp_debt"] = fit.dti / 100 * fit.annual_inc / 12
print("payment-to-income > 50%:", (fit.pti > .5).sum(), "| > 100%:", (fit.pti > 1).sum()); print(fit.pti.describe(percentiles=[.5, .99]).round(3))
out = {}
for k, d in fit.groupby("verification_status"):
    out[k] = {"share_grade_A_B": d.grade.isin(["A", "B"]).mean(), "median_loan": d.loan_amnt.median(), "median_int_rate": d.int_rate.median(),
              "spearman_income_vs_limit": d.annual_inc.corr(d.tot_hi_cred_lim, method="spearman"),
              "spearman_impdebt_vs_revolbal": d.imp_debt.corr(d.revol_bal, method="spearman")}
pd.DataFrame(out).T.round(3)

In [ ]:
# Assess whether missing borrower information is associated with loan default and identify related data-quality issues.
from scipy.stats import chi2_contingency
for c in ["mths_since_last_delinq", "emp_length", "emp_title"]:
    m = fit[c].isna(); p = chi2_contingency(pd.crosstab(m, fit.defaulted))[1]
    print(f"{c:24} missing {m.mean():.3f} | default if missing {fit.defaulted[m].mean():.3f} vs present {fit.defaulted[~m].mean():.3f} | chi2 p = {p:.4f}")
m = fit.mths_since_last_delinq.isna()
print("delinq months missing but delinq_2yrs > 0:", (m & (fit.delinq_2yrs > 0)).sum(), "of", m.sum())
print("emp_length missing and emp_title missing:", (fit.emp_length.isna() & fit.emp_title.isna()).sum(), "of", fit.emp_length.isna().sum())

In [ ]:
# Compare training and holdout features and identify potential target leakage using individual feature AUC.
hold = pd.read_csv("../data/holdout_applications.csv", low_memory=False)
print("train-only columns:", sorted(set(df.columns) - set(hold.columns)))
print("holdout-only columns:", sorted(set(hold.columns) - set(df.columns)))
from sklearn.metrics import roc_auc_score
num = [c for c in fit.select_dtypes("number").columns if c not in ["defaulted", "pti", "imp_debt"]]
s = pd.Series({c: roc_auc_score(fit.defaulted, fit[c].fillna(fit[c].median())) for c in num})
(s.sub(.5).abs().add(.5)).sort_values(ascending=False).round(3).head(15)

In [ ]:
# application-time features add some feature 
# Engineer application-time financial and credit-history features and evaluate their individual predictive power.
import json
from sklearn.metrics import roc_auc_score
spec = json.load(open("../reports/caps.json"))
def add_features(d, med_hist):
    f = pd.DataFrame(index=d.index)
    inc = d.annual_inc.clip(upper=spec["annual_inc"]["cap_p99.5"])
    f["credit_history_months"] = ((d.issue_d.dt.year - d.earliest_cr_line.dt.year) * 12 + (d.issue_d.dt.month - d.earliest_cr_line.dt.month)).fillna(med_hist)
    f["loan_to_income"] = d.loan_amnt / inc
    f["installment_to_income"] = d.installment * 12 / inc
    f["fico_mid"] = (d.fico_range_low + d.fico_range_high) / 2
    f["never_delinquent"] = d.mths_since_last_delinq.isna().astype(int)
    f["bc_util_missing"] = d.bc_util.isna().astype(int)
    return f
hist = (fit.issue_d.dt.year - fit.earliest_cr_line.dt.year) * 12 + (fit.issue_d.dt.month - fit.earliest_cr_line.dt.month)
spec["features"] = {"credit_history_months_fill": float(hist.median())}
json.dump(spec, open("../reports/caps.json", "w"), indent=1)
F = add_features(fit, spec["features"]["credit_history_months_fill"])
print(F.describe().T[["count", "mean", "50%", "min", "max"]].round(2))
auc = lambda s: round(abs(roc_auc_score(fit.defaulted, s) - .5) + .5, 3)
print("new:", {c: auc(F[c]) for c in F}); print("parents:", {c: auc(fit[c]) for c in ["loan_amnt", "annual_inc", "installment", "fico_range_low"]})
